# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()

print(f"{metadata_json['name']}:\n{metadata_json['description']}\n")
print(f"Cite As: {metadata_json['citeAs'] if 'citeAs' in metadata_json else 'N/A'}")
print(f"License: {metadata_json['license'] if 'license' in metadata_json else 'N/A'}")

## 2. Data Overview
Review available record sets and their fields and columns, referencing all by their `@id` (as per Croissant specification).

We will print out all record sets defined in the dataset, list their `@id`, and enumerate their available fields (columns) and their respective `@id`s.

In [ ]:
# Inspect all record sets and fields by @id
record_sets_info = []
for record_set in dataset.metadata.record_sets:
    print(f"RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    field_ids = []
    for field in record_set.fields:
        print(f"    Field name: {field.name}")
        print(f"      @id: {field.id}")
        print(f"      DataType: {getattr(field, 'data_type', '')}")
        field_ids.append(field.id)
    record_sets_info.append({'id': record_set.id, 'name': record_set.name, 'fields': field_ids})
    print()

print("Summary: Found the following record sets in the dataset (by @id):")
for rs in record_sets_info:
    print(f"  - {rs['id']}  (name: {rs['name']})")

## 3. Data Extraction
Load data from all defined record sets into DataFrames for analysis. All keys and lookups use the record set and field `@id` per the Croissant specification (see above).

In [ ]:
# Extract data for all available record sets
dataframes = {}
for record_set in record_sets_info:
    rs_id = record_set['id']
    # Retrieve records for this record set
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} rows for record set {rs_id} (name: {record_set['name']})")
        print(f"Columns (@id-names): {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No data found for record set {rs_id} (name: {record_set['name']})")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps on the main data record set. All entities are referenced by `@id`.

Below, we select a representative numeric field for filtering, normalization, and grouping. **Replace with actual field/column `@id`s for this dataset as listed above.**

In [ ]:
# --- Select primary record set and fields by @id as listed above ---
# Example: Use the first available record set with tabular clinical data
primary_record_set_id = None
numeric_field_id = None
group_field_id = None

# Try to find a numeric field for demo; you may need to adjust below based on printed column ids
for rs in record_sets_info:
    if rs['id'] in dataframes:
        df = dataframes[rs['id']]
        # Check which columns are numeric
        for col in df.columns:
            # Try to select float/int columns with non-null values
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                primary_record_set_id = rs['id']
                # Select a groupable field (string/object) that's not the same as the numeric field
                for col2 in df.columns:
                    if col2 != numeric_field_id and df[col2].dtype == object:
                        group_field_id = col2
                        break
                break
        if numeric_field_id is not None:
            break

if primary_record_set_id is None:
    raise RuntimeError("Could not find a suitable numeric field for EDA. Please inspect your dataset.")
print(f"Using record set: {primary_record_set_id}\nNumeric field: {numeric_field_id}\nGroup field: {group_field_id}")

df = dataframes[primary_record_set_id]
threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0  # Set threshold to mean as example
filtered_df = df[df[numeric_field_id] > threshold] if not df[numeric_field_id].isnull().all() else df
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
display(filtered_df.head())

# Normalize numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by categorical (group) field and show aggregate
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df)
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions and field relationships for the selected numeric and group fields.

Example: Histogram, boxplot and group-wise bar plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(6, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by group field
if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(8, 6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Group field not available for boxplot.")

## 6. Conclusion
We demonstrated how to load, explore, and process the FAIR² clinical dataset using `mlcroissant`, referencing all entities by their `@id` for robust schema navigation. This notebook provides a foundation for further statistical and machine learning analyses on the cohort.

Key findings and next steps:
- The dataset includes comprehensive clinicopathological and molecular features of second primary colorectal cancer survivors.
- Demonstrated selection and transformation of numeric fields, and group-wise analysis via field `@id`s.
- Visualized field distributions and group relationships to initiate hypothesis generation.

For deeper analysis (e.g., survival modeling, prediction), consult medical experts and use domain-relevant groupings and outcomes.